# Synthetic Slurm Pipeline Debug

One-sample local run for checking the Slurm pipeline phases before submitting the full batch.

In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

SYNTHETIC_ROOT = Path(r"/uufs/chpc.utah.edu/common/home/u0914269/clement/projects/20260624_methylseg/analysis/02_synthetic_analysis")
SLURM_CODE_DIR = SYNTHETIC_ROOT / "slurm_code"
for path in [SYNTHETIC_ROOT, SLURM_CODE_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import pipeline_config as cfg
import aggregate_metrics as agg
import synthetic_analysis_helpers as sah

DEBUG_ROOT = SYNTHETIC_ROOT / "debug_slurm_pipeline_run"
path_map = cfg.ensure_base_dirs(DEBUG_ROOT)
records = cfg.included_records()[:1]

FORCE_RECREATE = False
N_TOOL_JOBS = 8
N_SAMPLE_PROCS = 8
LOCAL_MAX_TOTAL_JOBS = 64
N_PROCS = 8
BOUNDARY_TOLERANCE_BP = 10_000



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Healthy PMD Calls

In [3]:
config_pairs = sah.write_comparator_configs(records, path_map["healthy_pmds"] / "configs")
comparator_class = sah.get_methyl_tool_comparator_class(cfg.COMPARATOR_DIR)

for record, config_path in config_pairs:
    comparator = comparator_class(
        config_file=str(config_path),
        out_dir=str(path_map["healthy_pmds"]),
        force_recreate=FORCE_RECREATE,
        n_jobs=N_TOOL_JOBS,
    )
    comparator.run()

config_pairs

[2026-06-25 18:01:20] Building shared prep artifacts in /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/02_synthetic_analysis/debug_slurm_pipeline_run/healthy_pmds/comparison/synthetic_WGBS_colon_primary_normal_1_hg38/shared_prep
[2026-06-25 18:01:26] Command gunzip -c /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/data/methylation_data/WGBS_colon-primary-normal_1_meth.bed.gz | Output: /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/02_synthetic_analysis/debug_slurm_pipeline_run/healthy_pmds/comparison/synthetic_WGBS_colon_primary_normal_1_hg38/shared_prep/..wgbs.tsv.tmp.3447717.1782432080200046959.unsorted.tmp.3447717.1782432080200143582 | Error: /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/02_synthetic_analysis/debug_slurm_pipeline_run/healthy_pmds/comparison/synthetic_WGBS_colon_primary_normal_1_hg38/logs/job_logs/gunzip_4853488.stderr | time

[({'source_file': PosixPath('/uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/data/methylation_data/WGBS_colon-primary-normal_1_meth.bed.gz'),
   'source_genome': 'hg38',
   'source_kind': 'wgbs_bed_gz',
   'include': True,
   'synthetic_sample_id': 'synthetic_WGBS_colon_primary_normal_1_hg38'},
  PosixPath('/uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/02_synthetic_analysis/debug_slurm_pipeline_run/healthy_pmds/configs/synthetic_WGBS_colon_primary_normal_1_hg38.yaml'))]

## 2. Healthy Backgrounds

In [4]:
sample_results, background_manifest_df, background_manifest_path = sah.build_healthy_background_references(
    records=records,
    healthy_pmd_root=path_map["healthy_pmds"],
    output_dir=path_map["backgrounds"],
    min_beta=0.2,
    max_beta=0.8,
    n_procs=N_PROCS,
)
background_validation_df = sah.validate_background_manifest(
    background_manifest_path,
    healthy_pmd_root=path_map["healthy_pmds"],
)
background_validation_df

,synthetic_sample_id,n_pmds_to_remove,n_modified_cpg_rows
0,synthetic_WGBS_colon_primary_normal_1_hg38,4024,19648113


In [5]:
sample_id = background_manifest_df["synthetic_sample_id"].iloc[0]
sample_id

'synthetic_WGBS_colon_primary_normal_1_hg38'

## 3. Inject PMDs

In [6]:
injected_manifest_df, injected_manifest_path = sah.build_injected_manifest(
    background_manifest=background_manifest_path,
    output_dir=path_map["injected"],
    config=sah.load_default_pmd_config(),
    selected_sample_ids=[sample_id],
    overwrite=FORCE_RECREATE,
    n_procs=N_PROCS,
)
injected_validation_df = sah.validate_injected_manifest(injected_manifest_path)
injected_validation_df

,synthetic_sample_id,n_truth_regions,n_truth_cpg_rows,n_modified_rows,mean_background_beta,mean_synthetic_beta
0,synthetic_WGBS_colon_primary_normal_1_hg38_inj...,159,1311748,1192669,0.875613,0.854822


## 4. Recovery

In [7]:
sample_id = injected_manifest_df["synthetic_sample_id"].iloc[0]
sample_id

'synthetic_WGBS_colon_primary_normal_1_hg38_injected_pmds'

In [ ]:
recovery_run = sah.run_synthetic_recovery(
    injected_manifest=injected_manifest_df,
    recovery_output_dir=path_map["recovery"],
    comparator_dir=cfg.COMPARATOR_DIR,
    selected_sample_ids=[sample_id],
    force_recreate=FORCE_RECREATE,
    n_jobs=N_TOOL_JOBS,
)
recovery_run["completed_samples"]

[2026-06-25 20:01:19] Building shared prep artifacts in /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/02_synthetic_analysis/debug_slurm_pipeline_run/synthetic_recovery/tool_results/comparison/synthetic_WGBS_colon_primary_normal_1_hg38_injected_pmds/shared_prep
[2026-06-25 20:01:19] Command cp /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/02_synthetic_analysis/debug_slurm_pipeline_run/injected_pmd_samples/synthetic_files/synthetic_WGBS_colon_primary_normal_1_hg38_injected_pmds.synthetic.tsv /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/02_synthetic_analysis/debug_slurm_pipeline_run/synthetic_recovery/tool_results/comparison/synthetic_WGBS_colon_primary_normal_1_hg38_injected_pmds/shared_prep/.wgbs.tsv.tmp.3447717.1782439279228142998 | Output: /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/02_synthetic_analysis/debug_slurm_pipeline_run

## 5. Metrics

In [ ]:
truth_region_bed= "/uufs/chpc.utah.edu/common/home/u0914269/clement/projects/20260624_methylseg/results/02_synthetic_analysis/injected_pmd_samples/truth_regions/synthetic_GSM5652176_Adipocytes_Z000000T7_hg19_injected_pmds.truth.bed"
recovered_region_bed= "/uufs/chpc.utah.edu/common/home/u0914269/clement/projects/20260624_methylseg/results/02_synthetic_analysis/synthetic_recovery/tool_results/methylseg/synthetic_GSM5652176_Adipocytes_Z000000T7_hg19_injected_pmds/out/wgbs/summary_files/segments_cleaned_PMD.bed"

In [ ]:
# TODO: CODEX START HERE
from pybedtools import BedTool
import pandas as pd
import numpy as np


def safe_divide(numerator, denominator):
    return float(numerator) / float(denominator) if denominator else 0.0


def f1_score_from_precision_recall(precision, recall):
    return (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0


def tag_bed(bed_file, prefix):
    bt = BedTool(bed_file).sort()
    lines = []
    for i, iv in enumerate(bt, start=1):
        lines.append("\t".join([
            iv.chrom,
            str(iv.start),
            str(iv.end),
            f"{prefix}{i}",
            str(int(iv.end) - int(iv.start))
        ]))
    return BedTool("\n".join(lines), from_string=True).sort()


def tagged_bed_to_df(tagged_bed: BedTool, prefix: str) -> pd.DataFrame:
    cols = ["chrom", "start", "end", f"{prefix}_id", f"{prefix}_len"]
    df = tagged_bed.to_dataframe(names=cols)
    for col in ["start", "end", f"{prefix}_len"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype(int)
    return df.rename(columns={
        "chrom": f"{prefix}_chrom",
        "start": f"{prefix}_start",
        "end": f"{prefix}_end",
    })


def pairwise_wao(a: BedTool, b: BedTool) -> pd.DataFrame:
    overlap_bed = a.intersect(b, wao=True)

    cols = [
        "a_chrom", "a_start", "a_end", "a_id", "a_len",
        "b_chrom", "b_start", "b_end", "b_id", "b_len",
        "overlap_bp",
    ]

    df = overlap_bed.to_dataframe(names=cols)

    for col in ["a_start", "a_end", "a_len", "b_start", "b_end", "b_len", "overlap_bp"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df[df["overlap_bp"] > 0].copy()

    for col in ["a_start", "a_end", "a_len", "b_start", "b_end", "b_len", "overlap_bp"]:
        df[col] = df[col].astype(int)

    return df


def summarize_resolution(pair_df, threshold_col, n_truth, n_recalled, truth_ids):
    if pair_df.empty:
        return {
            "truth_hits": 0,
            "recalled_hits": 0,
            "recall": 0.0,
            "precision": 0.0,
            "f1": 0.0,
            "per_region_recall_fraction": np.nan if n_truth == 0 else 0.0,
            "mabe": np.nan,
        }

    eligible = pair_df.loc[pair_df[threshold_col]].copy()
    truth_hits = eligible["truth_id"].nunique() if not eligible.empty else 0
    recall_hits = eligible["recall_id"].nunique() if not eligible.empty else 0
    recall = safe_divide(truth_hits, n_truth)
    precision = safe_divide(recall_hits, n_recalled)

    if n_truth == 0:
        per_region_recall_fraction = np.nan
        mabe = np.nan
    elif eligible.empty:
        per_region_recall_fraction = 0.0
        mabe = np.nan
    else:
        truth_order = {truth_id: idx for idx, truth_id in enumerate(truth_ids)}
        best_matches = (
            eligible.sort_values(
                ["truth_id", "truth_pct_overlap", "slop_overlap_bp", "recall_id"],
                ascending=[True, False, False, True],
            )
            .drop_duplicates(subset=["truth_id"], keep="first")
            .reset_index(drop=True)
        )
        per_truth_fraction = np.zeros(n_truth, dtype=float)
        for row in best_matches.itertuples(index=False):
            per_truth_fraction[truth_order[row.truth_id]] = min(1.0, float(row.truth_pct_overlap))
        per_region_recall_fraction = float(per_truth_fraction.mean())
        mabe = float(best_matches["mabe_bp"].mean())

    return {
        "truth_hits": int(truth_hits),
        "recalled_hits": int(recall_hits),
        "recall": recall,
        "precision": precision,
        "f1": f1_score_from_precision_recall(precision, recall),
        "per_region_recall_fraction": per_region_recall_fraction,
        "mabe": mabe,
    }


def score(truth_bed_file, recalled_bed_file, genome_file, tol):
    """Return region-level scoring at any, majority, and super-majority resolutions."""

    truth = tag_bed(truth_bed_file, "T")
    recalled_raw = BedTool(recalled_bed_file).sort()
    recalled = tag_bed(recalled_raw.fn, "R")
    truth_meta = tagged_bed_to_df(truth, "truth")
    recall_meta = tagged_bed_to_df(recalled, "recall")
    truth_slop = truth.slop(g=genome_file, b=tol).sort()

    orig_pairs = pairwise_wao(truth, recalled).rename(columns={
        "a_id": "truth_id",
        "b_id": "recall_id",
        "overlap_bp": "orig_overlap_bp",
    })

    slop_pairs = pairwise_wao(truth_slop, recalled).rename(columns={
        "a_id": "truth_id",
        "b_id": "recall_id",
        "overlap_bp": "slop_overlap_bp",
    })

    if slop_pairs.empty:
        pair_df = pd.DataFrame(columns=[
            "truth_id", "recall_id", "slop_overlap_bp", "truth_chrom", "truth_start", "truth_end",
            "truth_len", "recall_chrom", "recall_start", "recall_end", "recall_len", "orig_overlap_bp",
            "left_slop_bp", "right_slop_bp", "used_slop_bp", "denom", "truth_pct_overlap",
            "start_error_bp", "end_error_bp", "mabe_bp", "hit_any", "hit_majority", "hit_super_majority",
        ])
    else:
        pair_df = slop_pairs.merge(
            truth_meta,
            on="truth_id",
            how="left",
        ).merge(
            recall_meta,
            on="recall_id",
            how="left",
        ).merge(
            orig_pairs[["truth_id", "recall_id", "orig_overlap_bp"]],
            on=["truth_id", "recall_id"],
            how="left",
        )
        pair_df["orig_overlap_bp"] = pair_df["orig_overlap_bp"].fillna(0).astype(int)
        pair_df["left_slop_bp"] = (pair_df["truth_start"] - pair_df["recall_start"]).clip(lower=-tol, upper=tol)
        pair_df["right_slop_bp"] = (pair_df["recall_end"] - pair_df["truth_end"]).clip(lower=-tol, upper=tol)
        pair_df["used_slop_bp"] = pair_df["left_slop_bp"] + pair_df["right_slop_bp"]
        pair_df["denom"] = pair_df["truth_len"] + pair_df["used_slop_bp"]
        pair_df["truth_pct_overlap"] = np.where(
            pair_df["denom"] > 0,
            pair_df["slop_overlap_bp"] / pair_df["denom"],
            0.0,
        )
        pair_df["start_error_bp"] = (pair_df["recall_start"] - pair_df["truth_start"]).abs()
        pair_df["end_error_bp"] = (pair_df["recall_end"] - pair_df["truth_end"]).abs()
        pair_df["mabe_bp"] = (pair_df["start_error_bp"] + pair_df["end_error_bp"]) / 2.0
        pair_df["hit_any"] = pair_df["slop_overlap_bp"] >= 1
        pair_df["hit_majority"] = pair_df["truth_pct_overlap"] >= 0.50
        pair_df["hit_super_majority"] = pair_df["truth_pct_overlap"] >= 0.75

    truth_ids = truth_meta["truth_id"].tolist()
    recall_ids = recall_meta["recall_id"].tolist()
    n_truth = len(truth_ids)
    n_recalled = len(recall_ids)

    summary = {
        "any": summarize_resolution(pair_df, "hit_any", n_truth, n_recalled, truth_ids),
        "majority": summarize_resolution(pair_df, "hit_majority", n_truth, n_recalled, truth_ids),
        "super_majority": summarize_resolution(pair_df, "hit_super_majority", n_truth, n_recalled, truth_ids),
    }

    return {
        "truth": truth,
        "truth_slop": truth_slop,
        "recalled": recalled,
        "truth_meta": truth_meta,
        "recall_meta": recall_meta,
        "pair_detail": pair_df,
        "summary": summary,
    }


def cpg_level_metrics(cpgs_bed, truth_bed_file, recalled_bed_file, genome_file, tol=0):
    cpgs = cpgs_bed
    truth = BedTool(truth_bed_file).sort()
    recalled = BedTool(recalled_bed_file).sort()

    if tol > 0:
        truth_slop = truth.slop(g=genome_file, b=tol).sort()
        tol_band = truth_slop.subtract(truth, A=False).sort()
        scorable_cpgs = cpgs.subtract(tol_band, A=True)
    else:
        scorable_cpgs = cpgs

    truth_pos = scorable_cpgs.intersect(truth, u=True)
    pred_pos = scorable_cpgs.intersect(recalled, u=True)
    tp = truth_pos.intersect(recalled, u=True)

    n_cpgs = int(scorable_cpgs.count())
    n_truth_pos = int(truth_pos.count())
    n_pred_pos = int(pred_pos.count())
    n_tp = int(tp.count())

    fp = int(n_pred_pos - n_tp)
    fn = int(n_truth_pos - n_tp)
    tn = int(n_cpgs - (n_truth_pos + n_pred_pos - n_tp))

    precision = safe_divide(n_tp, n_tp + fp)
    recall = safe_divide(n_tp, n_tp + fn)
    specificity = safe_divide(tn, tn + fp)
    accuracy = safe_divide(n_tp + tn, n_cpgs)

    return {
        "bp_tp": n_tp,
        "bp_fp": fp,
        "bp_fn": fn,
        "bp_tn": tn,
        "bp_precision": precision,
        "bp_recall": recall,
        "bp_f1": f1_score_from_precision_recall(precision, recall),
        "bp_jaccard": safe_divide(n_tp, n_tp + fp + fn),
        "bp_specificity": specificity,
        "bp_accuracy": accuracy,
    }


def compute_fragmentation_absorption(pair_detail, truth_ids, recall_ids):
    if truth_ids:
        truth_counts = np.zeros(len(truth_ids), dtype=float)
        if not pair_detail.empty:
            truth_order = {truth_id: idx for idx, truth_id in enumerate(truth_ids)}
            counts = pair_detail.loc[pair_detail["hit_any"]].groupby("truth_id")["recall_id"].nunique()
            for truth_id, count in counts.items():
                truth_counts[truth_order[truth_id]] = float(count)
        fragmentation = float(truth_counts.mean())
    else:
        fragmentation = np.nan

    if recall_ids:
        recall_counts = np.zeros(len(recall_ids), dtype=float)
        if not pair_detail.empty:
            recall_order = {recall_id: idx for idx, recall_id in enumerate(recall_ids)}
            counts = pair_detail.loc[pair_detail["hit_any"]].groupby("recall_id")["truth_id"].nunique()
            for recall_id, count in counts.items():
                recall_counts[recall_order[recall_id]] = float(count)
        absorption = float(recall_counts.mean())
    else:
        absorption = np.nan

    return {
        "fragmentation": fragmentation,
        "absorption": absorption,
        "fragmentation_distance_from_1": abs(fragmentation - 1.0) if not pd.isna(fragmentation) else np.nan,
        "absorption_distance_from_1": abs(absorption - 1.0) if not pd.isna(absorption) else np.nan,
    }


def false_positive_regions(recalled_bed, truth_slop_bed):
    false_bed = recalled_bed.intersect(truth_slop_bed, v=True)
    if false_bed.count() == 0:
        return pd.DataFrame(columns=["chrom", "start", "end"])
    false_df = false_bed.to_dataframe(names=["chrom", "start", "end", "recall_id", "recall_len"])
    for col in ["start", "end", "recall_len"]:
        false_df[col] = pd.to_numeric(false_df[col], errors="coerce").astype(int)
    return false_df[["chrom", "start", "end"]].reset_index(drop=True)


def compute_false_region_betas(cpg_df, false_region_df):
    if false_region_df.empty:
        return pd.DataFrame(columns=["chrom", "start", "end", "region_length_bp", "n_region_cpgs", "mean_beta"])

    false_lines = []
    for i, row in enumerate(false_region_df.itertuples(index=False), start=1):
        false_lines.append("\t".join([
            row.chrom,
            str(int(row.start)),
            str(int(row.end)),
            f"F{i}",
            str(int(row.end) - int(row.start)),
        ]))
    false_bed = BedTool("\n".join(false_lines), from_string=True).sort()

    cpg_meta = cpg_df[["chrom", "start", "end", "beta"]].copy()
    cpg_meta = cpg_meta.rename(columns={"beta": "cpg_beta"})
    cpg_meta["cpg_id"] = [f"C{i}" for i in range(1, len(cpg_meta) + 1)]
    cpg_lines = []
    for row in cpg_meta.itertuples(index=False):
        cpg_lines.append("\t".join([
            row.chrom,
            str(int(row.start)),
            str(int(row.end)),
            row.cpg_id,
            "1",
        ]))
    cpg_bed = BedTool("\n".join(cpg_lines), from_string=True).sort()

    overlap_df = pairwise_wao(false_bed, cpg_bed).rename(columns={"a_id": "false_region_id", "b_id": "cpg_id"})
    false_meta = tagged_bed_to_df(false_bed, "false_region").rename(columns={
        "false_region_chrom": "chrom",
        "false_region_start": "start",
        "false_region_end": "end",
        "false_region_id": "false_region_id",
        "false_region_len": "region_length_bp",
    })[["false_region_id", "chrom", "start", "end", "region_length_bp"]]

    if overlap_df.empty:
        false_meta["n_region_cpgs"] = 0
        false_meta["mean_beta"] = np.nan
        return false_meta.drop(columns=["false_region_id"]).reset_index(drop=True)

    overlap_df = overlap_df.merge(cpg_meta[["cpg_id", "cpg_beta"]], on="cpg_id", how="left")
    summary_df = (
        overlap_df.groupby("false_region_id", as_index=False)
        .agg(n_region_cpgs=("cpg_id", "nunique"), mean_beta=("cpg_beta", "mean"))
    )
    result_df = false_meta.merge(summary_df, on="false_region_id", how="left")
    result_df["n_region_cpgs"] = pd.to_numeric(result_df["n_region_cpgs"], errors="coerce").fillna(0).astype(int)
    return result_df.drop(columns=["false_region_id"]).reset_index(drop=True)


def run_metrics(truth_bed_file, recalled_bed_file, sample_file, genome_file, tol=0):
    cpg_df = pd.read_csv(
        sample_file,
        sep="\t",
        skiprows=1,
        names=["chrom", "start", "end", "beta"],
        dtype={"chrom": str, "start": int, "end": int, "beta": float},
    )
    cpgs = BedTool.from_dataframe(cpg_df[["chrom", "start", "end"]]).sort()
    print("Computing region-level metrics...")
    region_metrics = score(
        truth_bed_file=truth_bed_file,
        recalled_bed_file=recalled_bed_file,
        genome_file=genome_file,
        tol=tol,
    )
    print("Computing CpG-level metrics...")
    cpg_metrics = cpg_level_metrics(
        cpgs_bed=cpgs,
        truth_bed_file=truth_bed_file,
        recalled_bed_file=recalled_bed_file,
        genome_file=genome_file,
        tol=tol,
    )

    truth_ids = region_metrics["truth_meta"]["truth_id"].tolist()
    recall_ids = region_metrics["recall_meta"]["recall_id"].tolist()
    print("Computing fragmentation and absorption metrics...")
    global_metrics = compute_fragmentation_absorption(
        region_metrics["pair_detail"],
        truth_ids,
        recall_ids,
    )
    print("Computing false positive region metrics...")
    false_region_df = false_positive_regions(region_metrics["recalled"], region_metrics["truth_slop"])
    false_beta_df = compute_false_region_betas(cpg_df, false_region_df)

    bp_metrics = {
        **cpg_metrics,
        "bp_per_region_recall_fraction": region_metrics["summary"]["any"]["per_region_recall_fraction"],
        "bp_mabe": region_metrics["summary"]["any"]["mabe"],
    }

    metric_row = {
        "boundary_tolerance_bp": int(tol),
        "n_truth_regions": int(len(truth_ids)),
        "n_predicted_regions": int(len(recall_ids)),
        **bp_metrics,
        **{
            f"region_any_{key}": value
            for key, value in region_metrics["summary"]["any"].items()
            if key in {"precision", "recall", "f1", "per_region_recall_fraction", "mabe"}
        },
        **{
            f"region_majority_{key}": value
            for key, value in region_metrics["summary"]["majority"].items()
            if key in {"precision", "recall", "f1", "per_region_recall_fraction", "mabe"}
        },
        **{
            f"region_super_majority_{key}": value
            for key, value in region_metrics["summary"]["super_majority"].items()
            if key in {"precision", "recall", "f1", "per_region_recall_fraction", "mabe"}
        },
        **global_metrics,
        "avg_false_pmd_beta": float(false_beta_df["mean_beta"].mean()) if not false_beta_df.empty else np.nan,
        "mean_false_pmds_called": int(len(false_region_df)),
    }

    return {
        "pair_detail": region_metrics["pair_detail"],
        "summary": region_metrics["summary"],
        "bp_metrics": bp_metrics,
        "global_metrics": {
            **global_metrics,
            "avg_false_pmd_beta": metric_row["avg_false_pmd_beta"],
            "mean_false_pmds_called": metric_row["mean_false_pmds_called"],
        },
        "false_positive_regions": false_region_df,
        "false_positive_beta_by_region": false_beta_df,
        "metric_row": metric_row,
    }






In [ ]:
metrics = run_metrics(
    truth_bed_file=truth_region_bed,
    recalled_bed_file=recovered_region_bed,
    sample_file="/uufs/chpc.utah.edu/common/home/u0914269/clement/projects/20260624_methylseg/results/02_synthetic_analysis/synthetic_recovery/tool_results/methylseg/synthetic_GSM5652176_Adipocytes_Z000000T7_hg19_injected_pmds/prep/wgbs.beta",
    genome_file="/uufs/chpc.utah.edu/common/home/u0914269/clement/projects/20260624_methylseg/data/reference_data/hg38.chrom.sizes",
    tol=BOUNDARY_TOLERANCE_BP,
)
metrics

In [ ]:
import importlib
import synthetic_analysis_helpers as sah
importlib.reload(sah)

In [ ]:
metrics_result = sah.run_metrics(
    injected_manifest=injected_manifest_df,
    tool_results_dir=path_map["recovery_tool_results"],
    metrics_output_dir=path_map["metrics"],
    boundary_tolerance_bp=BOUNDARY_TOLERANCE_BP,
    n_sample_procs=1,
    n_tool_procs=N_TOOL_JOBS,
)

In [ ]:
from IPython.display import Markdown, display

per_sample_df = metrics_result["per_sample_df"]
per_tool_summary_df = metrics_result["per_tool_summary_df"]
false_beta_df = metrics_result["false_beta_df"]
sample_status_df = metrics_result["sample_status_df"]
tool_status_df = metrics_result["tool_status_df"]

print(f"Local debug settings: up to {N_SAMPLE_PROCS} sample processes with {N_TOOL_JOBS} tool-metric jobs each = up to {LOCAL_MAX_TOTAL_JOBS} active jobs")

figures = agg.write_metric_plots(
    metrics_result,
    path_map["plots"],
    display_inline=False,
)
print(f"Wrote {len(figures)} HTML plots to: {path_map['plots']}")

for figure_name, figure in figures.items():
    display(Markdown(f"### {figure_name}"))
    display(figure)

display(sample_status_df)
display(tool_status_df.head())
display(per_sample_df.head())
display(per_tool_summary_df)
